In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas", "numpy", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import re
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-mpnet-base-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 64 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()

def word_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))

df["len1_words"] = df["sentence1"].map(word_count)
df["len2_words"] = df["sentence2"].map(word_count)
df["avg_words"] = (df["len1_words"] + df["len2_words"]) / 2.0
df["len_gap_words"] = (df["len1_words"] - df["len2_words"]).abs()

filtered_df = df[
    df["len1_words"].between(6, 24)
    & df["len2_words"].between(6, 24)
    & (df["len_gap_words"] <= 8)
].copy()

q_low = filtered_df["avg_words"].quantile(0.35)
q_high = filtered_df["avg_words"].quantile(0.65)

subset_df = filtered_df[
    filtered_df["avg_words"].between(q_low, q_high, inclusive="both")
].copy()

subset_df = subset_df.sort_values(
    by=["avg_words", "len_gap_words", "sentence1", "sentence2"],
    ascending=[True, True, True, True],
).reset_index(drop=True)

print({
    "original_num_examples": int(len(df)),
    "filtered_num_examples": int(len(filtered_df)),
    "subset_num_examples": int(len(subset_df)),
    "avg_words_quantile_low": float(q_low),
    "avg_words_quantile_high": float(q_high),
    "columns": subset_df.columns.tolist(),
})
print(subset_df[["sentence1", "sentence2", "label", "len1_words", "len2_words", "avg_words", "len_gap_words"]].head(10))


In [ ]:
length_stats = {
    "len1_words_mean": float(subset_df["len1_words"].mean()),
    "len1_words_median": float(subset_df["len1_words"].median()),
    "len1_words_min": int(subset_df["len1_words"].min()),
    "len1_words_max": int(subset_df["len1_words"].max()),
    "len2_words_mean": float(subset_df["len2_words"].mean()),
    "len2_words_median": float(subset_df["len2_words"].median()),
    "len2_words_min": int(subset_df["len2_words"].min()),
    "len2_words_max": int(subset_df["len2_words"].max()),
    "avg_words_mean": float(subset_df["avg_words"].mean()),
    "avg_words_median": float(subset_df["avg_words"].median()),
    "avg_words_min": float(subset_df["avg_words"].min()),
    "avg_words_max": float(subset_df["avg_words"].max()),
    "len_gap_words_mean": float(subset_df["len_gap_words"].mean()),
    "len_gap_words_median": float(subset_df["len_gap_words"].median()),
}
print(length_stats)


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)


In [ ]:
sentences1 = subset_df["sentence1"].tolist()
sentences2 = subset_df["sentence2"].tolist()
labels = subset_df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)


In [ ]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic

results_df = subset_df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["signed_error"] = results_df["predicted_score_0_5"] - results_df["label"]
results_df["absolute_error"] = results_df["signed_error"].abs()

avg_len = results_df["avg_words"]
results_df["length_band"] = pd.cut(
    avg_len,
    bins=[-np.inf, 10, 14, 18, np.inf],
    labels=["short", "medium", "long", "very_long"],
    ordered=True,
)

bucket_summary_df = (
    results_df.groupby("length_band", observed=False)
    .agg(
        n=("label", "size"),
        avg_words_mean=("avg_words", "mean"),
        label_mean=("label", "mean"),
        pred_mean=("predicted_score_0_5", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("signed_error", lambda s: float(np.sqrt(np.mean(np.square(s))))),
        bias=("signed_error", "mean"),
    )
    .reset_index()
)

largest_errors_df = results_df.sort_values(
    ["absolute_error", "avg_words", "sentence1", "sentence2"],
    ascending=[False, True, True, True],
).reset_index(drop=True)

print({
    "pearson_correlation": float(pearson_corr),
    "spearman_correlation": float(spearman_corr),
    "mean_absolute_error": float(results_df["absolute_error"].mean()),
    "root_mean_squared_error": float(np.sqrt(np.mean(np.square(results_df["signed_error"])))),
})

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "absolute_error", "length_band"]].head(10))
print(bucket_summary_df)
print(largest_errors_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "signed_error", "absolute_error", "len1_words", "len2_words", "length_band"]].head(15))


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"original_num_examples: {len(df)}")
print(f"filtered_num_examples: {len(filtered_df)}")
print(f"subset_num_examples: {len(subset_df)}")
print(f"subset_rule: sentence1_words_and_sentence2_words_in_[6,24]_and_length_gap<=8_then_avg_words_quantile_slice_[0.35,0.65]")
print(f"avg_words_quantile_low: {q_low:.4f}")
print(f"avg_words_quantile_high: {q_high:.4f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"avg_len1_words: {subset_df['len1_words'].mean():.2f}")
print(f"avg_len2_words: {subset_df['len2_words'].mean():.2f}")
print(f"avg_length_gap_words: {subset_df['len_gap_words'].mean():.2f}")
print(f"mean_absolute_error: {results_df['absolute_error'].mean():.6f}")
print(f"root_mean_squared_error: {np.sqrt(np.mean(np.square(results_df['signed_error']))):.6f}")
print(f"max_absolute_error: {results_df['absolute_error'].max():.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")

top_error_examples = largest_errors_df[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "signed_error", "absolute_error", "len1_words", "len2_words", "length_band"
]].head(10)
print(top_error_examples.to_dict(orient="records"))
print(bucket_summary_df.to_dict(orient="records"))
